In [1]:
import os, json, time, statistics
from pathlib import Path
from pyspark.sql import SparkSession, functions as F
Path('results').mkdir(exist_ok=True)
Path('results/spark-events').mkdir(exist_ok=True)


In [2]:
s = (SparkSession.builder.master('local[2]').appName('hw22-skew')
     .config('spark.driver.memory', '1g').config('spark.executor.memory', '1g')
     .config('spark.sql.shuffle.partitions', '16').config('spark.sql.adaptive.enabled', 'false')
     .config('spark.sql.autoBroadcastJoinThreshold', '-1').config('spark.ui.showConsoleProgress', 'false')
     .config('spark.eventLog.enabled', 'true').config('spark.eventLog.compress', 'false')
     .config('spark.eventLog.dir', Path('results/spark-events').resolve().as_uri()).getOrCreate())
s.sparkContext.setLogLevel('ERROR')
app_id = s.sparkContext.applicationId
v = s.read.schema('user_uid long,element_uid long,watched_time long').option('header', True).csv('data/task2/skew_transactions.csv')
catalogue = json.loads(Path('data/task2/catalogue.json').read_text())
c = s.read.json(s.sparkContext.parallelize([json.dumps(dict(element_uid=int(k), **value)) for k, value in catalogue.items()], 2))
v.write.mode('overwrite').parquet('results/views')
c.write.mode('overwrite').parquet('results/catalogue')
v, c = s.read.parquet('results/views'), s.read.parquet('results/catalogue')
total = v.count()
freq = v.groupBy('element_uid').count().orderBy(F.desc('count'))
freq.withColumn('share', F.col('count') / total).show(10)
print('Просмотры:', total, 'каталог:', c.count())
print('NULL в ключах:', v.filter('element_uid IS NULL').count(), c.filter('element_uid IS NULL').count())
assert not c.groupBy('element_uid').count().filter('count > 1').count()
print('Ключ каталога уникален.')
print('Каталог Parquet, байт:', sum(f.stat().st_size for f in Path('results/catalogue').glob('*.parquet')))
hot = [r.element_uid for r in freq.filter(F.col('count') > total / 16).collect()]
print('Горячие ключи:', hot)


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/13 19:54:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+-----------+--------+--------------------+
|element_uid|   count|               share|
+-----------+--------+--------------------+
|       2714|20496000|  0.6815938921525317|
|        747|   61272|0.002037598602652...|
|       2639|   61119|0.002032510592040915|
|       3783|   55228|0.001836605556001...|
|       2245|   52942|0.001760584691566127|
|       6127|   52168|0.001734845343765...|
|      10061|   48993|0.001629260809827722|
|       3916|   48243|0.001604319581338...|
|       8771|   47480|0.001578946038222...|
|       3336|   45646|0.001517956420823...|
+-----------+--------+--------------------+
only showing top 10 rows

Просмотры: 30070692 каталог: 10200
NULL в ключах: 0 0


Ключ каталога уникален.
Каталог Parquet, байт: 604547


Горячие ключи: [2714]


In [3]:
def measurements(app_id):
    stages, executions, plans, tasks = {}, {}, {}, {}
    for path in Path('results/spark-events').glob(app_id + '*'):
        for line in path.open():
            e = json.loads(line)
            kind = e['Event']
            if kind == 'SparkListenerJobStart':
                props = e.get('Properties') or {}
                group = props.get('spark.jobGroup.id')
                if group:
                    stages.setdefault(group, set()).update(e['Stage IDs'])
                    if 'spark.sql.execution.id' in props:
                        executions.setdefault(group, set()).add(int(props['spark.sql.execution.id']))
            elif kind.endswith(('SparkListenerSQLExecutionStart', 'SparkListenerSQLAdaptiveExecutionUpdate')):
                plans[e['executionId']] = e.get('physicalPlanDescription', '')
            elif kind == 'SparkListenerTaskEnd' and e.get('Task End Reason', {}).get('Reason') == 'Success':
                m = e.get('Task Metrics', {})
                read = m.get('Shuffle Read Metrics', {})
                info = e['Task Info']
                tasks.setdefault(e['Stage ID'], []).append({
                    'shuffle_read': read.get('Remote Bytes Read', 0) + read.get('Local Bytes Read', 0),
                    'shuffle_write': m.get('Shuffle Write Metrics', {}).get('Shuffle Bytes Written', 0),
                    'memory_spill': m.get('Memory Bytes Spilled', 0),
                    'disk_spill': m.get('Disk Bytes Spilled', 0),
                    'task_ms': info['Finish Time'] - info['Launch Time']})
    rows = []
    for group, ids in stages.items():
        ts = [t for sid in ids for t in tasks.get(sid, [])]
        row = {'run': group, **{k: sum(t[k] for t in ts) for k in ['shuffle_read', 'shuffle_write', 'memory_spill', 'disk_spill']}}
        durations = [t['task_ms'] for t in ts]
        row.update(task_median_ms=statistics.median(durations) if durations else 0, task_max_ms=max(durations, default=0))
        rows.append(row)
        if group.endswith('_5') or group in ('baseline', 'broadcast', 'salt'):
            print(group)
            for eid in sorted(executions.get(group, [])):
                print(plans.get(eid, ''))
    import pandas as pd
    display(pd.DataFrame(rows))
    return rows

def same(a, b):
    return not a.exceptAll(b.select(a.columns)).limit(1).count() and not b.select(a.columns).exceptAll(a).limit(1).count()


In [4]:
sv = v.withColumn('salt', F.when(F.col('element_uid').isin(hot), F.pmod(F.monotonically_increasing_id(), F.lit(16))).otherwise(F.lit(0)))
sc = c.withColumn('salt', F.explode(F.when(F.col('element_uid').isin(hot), F.sequence(F.lit(0), F.lit(15))).otherwise(F.array(F.lit(0)))))
queries = {'baseline': v.join(c, 'element_uid', 'inner'),
           'broadcast': v.join(F.broadcast(c), 'element_uid', 'inner'),
           'salt': sv.hint('MERGE').join(sc.hint('MERGE'), ['element_uid', 'salt'], 'inner').drop('salt')}
rows, outputs = [], {}
for name, query in queries.items():
    query = query.select('element_uid', *[x for x in v.columns if x != 'element_uid'], *[x for x in c.columns if x != 'element_uid'])
    s.sparkContext.setJobGroup(name, name)
    start = time.perf_counter()
    query.write.mode('overwrite').parquet('results/' + name)
    seconds = time.perf_counter() - start
    s.sparkContext.setLocalProperty('spark.jobGroup.id', None)
    actual = s.read.parquet('results/' + name)
    rows.append({'strategy': name, 'seconds': seconds, 'rows': actual.count(), 'files': len(actual.inputFiles())})
    outputs[name] = actual
    print(name, round(seconds, 3), flush=True)
for name in ['broadcast', 'salt']:
    assert same(outputs['baseline'], outputs[name])
    print(name, 'полное равенство с учётом повторов: True')
import pandas as pd
display(pd.DataFrame(rows))
print('Дополнительных строк каталога:', sc.count() - c.count())
s.stop()


baseline 37.938


broadcast 24.584


salt 27.73


broadcast полное равенство с учётом повторов: True


salt полное равенство с учётом повторов: True


,strategy,seconds,rows,files
0,baseline,37.938203,30070692,16
1,broadcast,24.583741,30070692,3
2,salt,27.729795,30070692,16


Дополнительных строк каталога: 15


In [5]:
metrics = measurements(app_id)
print('Горячий ключ концентрирует большую часть просмотров в одном разделе SortMergeJoin. Выше приведены частоты и максимальное/медианное время задач.')
print('Broadcast применим: каталог мал и помещается в память; большая таблица не перераспределяется по горячему ключу.')
print('Salt=16 применяется только к ключам с долей выше 1/16. Каждому просмотру назначается одна соль, соответствующая строка каталога копируется на все 16 значений: каждый исходный матч сохраняется один раз. Цена — дополнительные строки каталога и shuffle.')


baseline
== Physical Plan ==
Execute InsertIntoHadoopFsRelationCommand (14)
+- WriteFiles (13)
   +- * Project (12)
      +- * SortMergeJoin Inner (11)
         :- * Sort (5)
         :  +- Exchange (4)
         :     +- * Filter (3)
         :        +- * ColumnarToRow (2)
         :           +- Scan parquet  (1)
         +- * Sort (10)
            +- Exchange (9)
               +- * Filter (8)
                  +- * ColumnarToRow (7)
                     +- Scan parquet  (6)


(1) Scan parquet 
Output [3]: [user_uid#48L, element_uid#49L, watched_time#50L]
Batched: true
Location: InMemoryFileIndex [file:/work/results/views]
PushedFilters: [IsNotNull(element_uid)]
ReadSchema: struct<user_uid:bigint,element_uid:bigint,watched_time:bigint>

(2) ColumnarToRow [codegen id : 1]
Input [3]: [user_uid#48L, element_uid#49L, watched_time#50L]

(3) Filter [codegen id : 1]
Input [3]: [user_uid#48L, element_uid#49L, watched_time#50L]
Condition : isnotnull(element_uid#49L)

(4) Exchange
Input [3]: 

,run,shuffle_read,shuffle_write,memory_spill,disk_spill,task_median_ms,task_max_ms
0,baseline,377828032,377828032,763361872,226659800,967,32049
1,broadcast,0,0,0,0,10168,22358
2,salt,409043532,409043532,629144400,163255004,2081,9289


Горячий ключ концентрирует большую часть просмотров в одном разделе SortMergeJoin. Выше приведены частоты и максимальное/медианное время задач.
Broadcast применим: каталог мал и помещается в память; большая таблица не перераспределяется по горячему ключу.
Salt=16 применяется только к ключам с долей выше 1/16. Каждому просмотру назначается одна соль, соответствующая строка каталога копируется на все 16 значений: каждый исходный матч сохраняется один раз. Цена — дополнительные строки каталога и shuffle.
